In [ ]:
"""Importing Libraries"""

#General
import pandas as pd
import numpy as np
import scipy
import scipy.io
import os
import zipfile
import matplotlib.pyplot as plt
from scipy import signal
import librosa
import random

#Deep Learning
import imgaug.augmenters as iaa
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.optimizers import Adam, SGD, RMSprop
from tensorflow.keras.layers import Input, Conv2D, Conv1D, MaxPooling1D, BatchNormalization, Dense, MaxPooling2D, Flatten, Dense, Dropout, concatenate, LSTM, Reshape, Concatenate, Activation, Permute, Multiply
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix
import seaborn as sns

from sklearn.svm import SVC

import gc

In [ ]:
"""Mounting Google Drive"""
from google.colab import drive
drive.mount('/content/drive', force_remount = True)

Mounted at /content/drive


In [ ]:
pip install nolds

In [ ]:
import numpy as np
import pandas as pd
from scipy.signal import spectrogram
from scipy.stats import entropy
from nolds import lyap_r, dfa
from scipy.signal import welch

In [ ]:
def calculate_fractal_dimensions(data):
    # Calculate fractal dimensions using Higuchi Fractal Dimension (HFD)
    # You can replace this with another method if desired
    hfd_values = [dfa(data[channel]) for channel in range(data.shape[0])]
    return hfd_values

def calculate_lyapunov_exponents(data):
    # Calculate Lyapunov exponents
    lyap_values = [lyap_r(data[channel], emb_dim=10) for channel in range(data.shape[0])]
    return lyap_values

def calculate_spectral_entropy(data, fs=500):
    # Calculate spectral entropy
    entropy_values = []
    for channel in range(data.shape[0]):
        f, Pxx = welch(data[channel], fs, nperseg=256)
        spectral_entropy = entropy(Pxx, base=2)
        entropy_values.append(spectral_entropy)
    return entropy_values

def calculate_power_bands(data, fs=500):
    # Calculate power bands (alpha, beta, theta, gamma)
    power_bands = []
    for channel in range(data.shape[0]):
        f, Pxx = welch(data[channel], fs, nperseg=256)
        alpha_power = np.sum(Pxx[(f >= 8) & (f <= 13)])
        beta_power = np.sum(Pxx[(f > 13) & (f <= 30)])
        theta_power = np.sum(Pxx[(f >= 4) & (f <= 7)])
        gamma_power = np.sum(Pxx[(f > 30) & (f <= 50)])
        power_bands.append([alpha_power, beta_power, theta_power, gamma_power])
    return power_bands

def calculate_power_band_ratios(power_bands):
    # Calculate power band ratios
    ratios = []
    for band in power_bands:
        total_power = np.sum(band)
        ratios.append([x / total_power for x in band])
    return ratios


In [ ]:
def extract_features(data, num_frames=50):
    num_samples = data.shape[1]
    num_channels = data.shape[0]

    # Calculate window size and overlap
    window_size = num_samples // num_frames
    overlap = window_size // 2
    print(window_size, flush=True)
    print(overlap, flush=True)

    Features = np.empty((18, 0))
    for i in range(0, num_samples - window_size + 1, window_size - overlap):
        frame = data[:, i:i+window_size]

        # Calculate features for each frame
        hfd_values = calculate_fractal_dimensions(frame)
        print(np.shape(hfd_values), flush=True)
        lyap_values = calculate_lyapunov_exponents(frame)
        print(np.shape(lyap_values), flush=True)
        entropy_values = calculate_spectral_entropy(frame)
        print(np.shape(entropy_values), flush=True)
        power_bands = calculate_power_bands(frame)
        print(np.shape(power_bands), flush=True)
        band_ratios = calculate_power_band_ratios(power_bands)
        print(np.shape(band_ratios), flush=True)

        combined_features = np.concatenate((np.array(hfd_values).reshape(-1, 1), np.array(lyap_values).reshape(-1, 1),
                                            np.array(entropy_values).reshape(-1, 1), np.array(power_bands),
                                            np.array(band_ratios)), axis=1)

        Features = np.concatenate((Features, combined_features), axis=1)

    return np.array(Features)

In [ ]:
num_frames = 50
print(num_frames, flush=True)
csv_folder = '/content/drive/MyDrive/Alzheimers Disease Dataset/CSV Files/'
output_folder = '/content/drive/MyDrive/Alzheimers Disease Dataset/Features/'


# Get a list of all CSV files in the folder
csv_files = [file for file in os.listdir(csv_folder) if file.endswith('.csv')]

csv_files.sort(key=lambda x: int(x.split('-')[-1].split('.')[0]))


print('CSV Files Extracted', flush=True)


# Iterate over each CSV file
for csv_file in csv_files:
    csv_file_path = os.path.join(csv_folder, csv_file)

    # Read data from CSV file
    df = pd.read_csv(csv_file_path)
    print('Data has been read', flush=True)
    features = extract_features(df.values, num_frames=num_frames)
    print(features.shape)

    csv_file_name = os.path.splitext(csv_file)[0]  # Get the filename without extension
    output_file_path = os.path.join(output_folder, f"{csv_file_name}_features.csv")

    # Save features to CSV file
    pd.DataFrame(features).to_csv(output_file_path, index=False, header=None)
    print(f"Features saved to: {output_file_path}")

50
CSV Files Extracted
Data has been read
7977
3988
(18,)


/usr/local/lib/python3.10/dist-packages/nolds/measures.py:240: RuntimeWarning: signal has very low mean frequency, setting min_tsep = 1994
  warnings.warn(msg.format(min_tsep), RuntimeWarning)
/usr/local/lib/python3.10/dist-packages/nolds/measures.py:240: RuntimeWarning: signal has very low mean frequency, setting min_tsep = 1994
  warnings.warn(msg.format(min_tsep), RuntimeWarning)
/usr/local/lib/python3.10/dist-packages/nolds/measures.py:240: RuntimeWarning: signal has very low mean frequency, setting min_tsep = 1994
  warnings.warn(msg.format(min_tsep), RuntimeWarning)
/usr/local/lib/python3.10/dist-packages/nolds/measures.py:240: RuntimeWarning: signal has very low mean frequency, setting min_tsep = 1994
  warnings.warn(msg.format(min_tsep), RuntimeWarning)
/usr/local/lib/python3.10/dist-packages/nolds/measures.py:240: RuntimeWarning: signal has very low mean frequency, setting min_tsep = 1994
  warnings.warn(msg.format(min_tsep), RuntimeWarning)
/usr/local/lib/python3.10/dist-pack

(18,)
(18,)
(18, 4)
(18, 4)
(18,)


/usr/local/lib/python3.10/dist-packages/nolds/measures.py:240: RuntimeWarning: signal has very low mean frequency, setting min_tsep = 1994
  warnings.warn(msg.format(min_tsep), RuntimeWarning)
/usr/local/lib/python3.10/dist-packages/nolds/measures.py:240: RuntimeWarning: signal has very low mean frequency, setting min_tsep = 1994
  warnings.warn(msg.format(min_tsep), RuntimeWarning)
/usr/local/lib/python3.10/dist-packages/nolds/measures.py:240: RuntimeWarning: signal has very low mean frequency, setting min_tsep = 1994
  warnings.warn(msg.format(min_tsep), RuntimeWarning)
/usr/local/lib/python3.10/dist-packages/nolds/measures.py:240: RuntimeWarning: signal has very low mean frequency, setting min_tsep = 1994
  warnings.warn(msg.format(min_tsep), RuntimeWarning)
/usr/local/lib/python3.10/dist-packages/nolds/measures.py:240: RuntimeWarning: signal has very low mean frequency, setting min_tsep = 1994
  warnings.warn(msg.format(min_tsep), RuntimeWarning)
/usr/local/lib/python3.10/dist-pack

(18,)
(18,)
(18, 4)
(18, 4)
(18,)


...

(18,)
(18,)
(18, 4)
(18, 4)
(18,)


/usr/local/lib/python3.10/dist-packages/nolds/measures.py:240: RuntimeWarning: signal has very low mean frequency, setting min_tsep = 2125
  warnings.warn(msg.format(min_tsep), RuntimeWarning)
/usr/local/lib/python3.10/dist-packages/nolds/measures.py:240: RuntimeWarning: signal has very low mean frequency, setting min_tsep = 2125
  warnings.warn(msg.format(min_tsep), RuntimeWarning)
/usr/local/lib/python3.10/dist-packages/nolds/measures.py:240: RuntimeWarning: signal has very low mean frequency, setting min_tsep = 2125
  warnings.warn(msg.format(min_tsep), RuntimeWarning)
/usr/local/lib/python3.10/dist-packages/nolds/measures.py:240: RuntimeWarning: signal has very low mean frequency, setting min_tsep = 2125
  warnings.warn(msg.format(min_tsep), RuntimeWarning)
/usr/local/lib/python3.10/dist-packages/nolds/measures.py:240: RuntimeWarning: signal has very low mean frequency, setting min_tsep = 2125
  warnings.warn(msg.format(min_tsep), RuntimeWarning)
/usr/local/lib/python3.10/dist-pack

(18,)
(18,)
(18, 4)
(18, 4)
(18,)


/usr/local/lib/python3.10/dist-packages/nolds/measures.py:240: RuntimeWarning: signal has very low mean frequency, setting min_tsep = 2125
  warnings.warn(msg.format(min_tsep), RuntimeWarning)
/usr/local/lib/python3.10/dist-packages/nolds/measures.py:240: RuntimeWarning: signal has very low mean frequency, setting min_tsep = 2125
  warnings.warn(msg.format(min_tsep), RuntimeWarning)
/usr/local/lib/python3.10/dist-packages/nolds/measures.py:240: RuntimeWarning: signal has very low mean frequency, setting min_tsep = 2125
  warnings.warn(msg.format(min_tsep), RuntimeWarning)
/usr/local/lib/python3.10/dist-packages/nolds/measures.py:240: RuntimeWarning: signal has very low mean frequency, setting min_tsep = 2125
  warnings.warn(msg.format(min_tsep), RuntimeWarning)
/usr/local/lib/python3.10/dist-packages/nolds/measures.py:240: RuntimeWarning: signal has very low mean frequency, setting min_tsep = 2125
  warnings.warn(msg.format(min_tsep), RuntimeWarning)
/usr/local/lib/python3.10/dist-pack

(18,)
(18,)
(18, 4)
(18, 4)
(18, 1089)
Features saved to: /content/drive/MyDrive/Alzheimers Disease Dataset/Features/Sub-sub-039_features.csv
